# RQ1 — por que os polígonos de especialistas são menos diversos entre si que a ciência cidadã (diagnóstico)

**Motivação.** `expert_polygon_redundancy_check.ipynb` encontrou um
achado inesperado: mesmo *entre* polígonos diferentes (ignorando a
redundância dentro de cada um), os píxeis de especialistas são mais
parecidos entre si (similaridade de cosseno média 0,778) do que as
presenças de ciência cidadã são entre si (0,613). Isto pode explicar a
curva plana melhor do que a redundância dentro do polígono — mas não
diz *porquê*. Duas hipóteses, não mutuamente exclusivas:

1. **Concentração geográfica:** os polígonos de especialistas cobrem
   uma área física menor/mais concentrada do que as observações de
   ciência cidadã, dentro do recorte-piloto.
2. **Concentração de habitat/uso do solo:** mesmo espalhados
   geograficamente, os polígonos podem cair sistematicamente num tipo
   de paisagem mais homogéneo do que a ciência cidadã (que inclui
   observações oportunistas perto de estradas, jardins, orlas
   urbanas, etc.).

Este notebook testa as duas, com uma experiência direta: recalcular a
similaridade entre pares de ciência cidadã **só dentro dos mesmos
blocos de 10 km onde há polígonos de especialistas** — se subir para
perto de 0,778, a explicação é geográfica; se ficar perto de 0,613
mesmo ali, há algo mais específico do habitat escolhido pelos
especialistas.


## 1. Carregar embeddings, presenças e blocos


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
cs_x, cs_y, cs_emb, cs_n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

expert_gdf = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
joined = gpd.sjoin(pilot_points, expert_gdf[["geometry"]], predicate="within", how="inner")
joined = joined[~joined.index.duplicated(keep="first")]
expert_idx = joined.index.values
expert_poly_id_all = joined["index_right"].values
expert_emb_all = pilot_emb[expert_idx]
valid = ~np.isnan(expert_emb_all).any(axis=1)
expert_x, expert_y = pilot_df["x"].values[expert_idx][valid], pilot_df["y"].values[expert_idx][valid]
expert_emb = expert_emb_all[valid]
expert_poly_id = expert_poly_id_all[valid]

cs_block = block_id(cs_x, cs_y)
expert_block = block_id(expert_x, expert_y)

print(f"citizen-science presences: {len(cs_emb):,}, across {len(np.unique(cs_block)):,} blocks (10 km)")
print(f"expert presence pixels: {len(expert_emb):,}, across {len(np.unique(expert_block)):,} blocks (10 km), {len(np.unique(expert_poly_id)):,} polygons")


## 2. Espalhamento geográfico: blocos, área do invólucro convexo, distância entre pares


In [ ]:
from shapely.geometry import MultiPoint

def spread_stats(x, y, label):
    hull = MultiPoint(list(zip(x, y))).convex_hull
    area_km2 = hull.area / 1e6
    rng = np.random.default_rng(2)
    n = len(x)
    ia = rng.integers(0, n, 5000)
    ib = rng.integers(0, n, 5000)
    dist = np.sqrt((x[ia] - x[ib]) ** 2 + (y[ia] - y[ib]) ** 2)
    print(f"{label}: convex hull area = {area_km2:,.1f} km2, "
          f"pairwise distance mean = {dist.mean() / 1000:.2f} km, median = {np.median(dist) / 1000:.2f} km")
    return dist

cs_dist = spread_stats(cs_x, cs_y, "citizen-science")
expert_dist = spread_stats(expert_x, expert_y, "expert")


## 3. Composição de uso do solo (COS 2023): especialistas vs. ciência cidadã

Mesmo padrão de junção espacial por bloco de 10 km de
`expert_polygons.ipynb` / `forest_only_background_check.ipynb`.


In [ ]:
def cos_group(code):
    if pd.isna(code):
        return "9 no COS coverage"
    if code.startswith("1"):
        return "1 artificial (buildings, industry, roads)"
    if code[0] in "234":
        return "2 agriculture and pasture"
    if code == "5.1.1.6":
        return "3 eucalyptus forest"
    if code == "5.1.1.7":
        return "4 acacia forest (COS)"
    if code.startswith("5.1.2"):
        return "5 pine forest"
    if code.startswith("5"):
        return "6 other forest"
    if code.startswith("6"):
        return "7 shrubland (matos)"
    return "8 water, wetlands, bare"


def cos_composition(x, y, label):
    pts = gpd.GeoDataFrame({"x": x, "y": y}, geometry=gpd.points_from_xy(x, y), crs=32629).to_crs(3763)
    blocks = pd.Series(list(zip(pts.geometry.x // 10_000, pts.geometry.y // 10_000)), index=pts.index)
    joined = []
    for (bx, by), d in pts.groupby(blocks):
        cos = gpd.read_file(COS, layer="COS2023v1", bbox=(bx * 1e4, by * 1e4, (bx + 1) * 1e4, (by + 1) * 1e4), columns=["COS23_n4_C"])
        joined.append(gpd.sjoin(d, cos, predicate="within", how="left").drop(columns="index_right"))
    pts = pd.concat(joined)
    pts = pts[~pts.index.duplicated()]
    pts["cos_group"] = pts["COS23_n4_C"].map(cos_group)
    counts = pts["cos_group"].value_counts(normalize=True).mul(100).round(1)
    counts.name = label
    return counts


comp = pd.concat([cos_composition(cs_x, cs_y, "citizen-science %"), cos_composition(expert_x, expert_y, "expert %")], axis=1).fillna(0.0)
comp.sort_index()


## 4. O teste direto: ciência cidadã, só dentro dos blocos com especialistas

Se a similaridade da ciência cidadã, restrita aos mesmos blocos de 10 km
onde há polígonos de especialistas, subir para perto de 0,778 (a
similaridade entre-polígonos de especialistas encontrada no
diagnóstico anterior), a explicação é geográfica. Se continuar perto
de 0,613 (o valor da ciência cidadã no distrito inteiro), os
especialistas estão a cobrir um habitat mais estreito mesmo dentro da
mesma área.


In [ ]:
def sample_cosine(emb_a, emb_b, n_pairs, rng):
    norm_a = emb_a / (np.linalg.norm(emb_a, axis=1, keepdims=True) + 1e-12)
    norm_b = emb_b / (np.linalg.norm(emb_b, axis=1, keepdims=True) + 1e-12)
    idx_a = rng.integers(0, len(norm_a), n_pairs)
    idx_b = rng.integers(0, len(norm_b), n_pairs)
    return (norm_a[idx_a] * norm_b[idx_b]).sum(axis=1)


rng = np.random.default_rng(1)
N_PAIRS = 5000

expert_blocks_unique = set(expert_block)
cs_in_expert_blocks = np.isin(cs_block, list(expert_blocks_unique))
cs_emb_restricted = cs_emb[cs_in_expert_blocks]
print(f"citizen-science presences inside the {len(expert_blocks_unique)} expert-containing blocks: "
      f"{cs_in_expert_blocks.sum():,} of {len(cs_emb):,}")

cs_sims_full = sample_cosine(cs_emb, cs_emb, N_PAIRS, rng)
cs_sims_restricted = sample_cosine(cs_emb_restricted, cs_emb_restricted, N_PAIRS, rng)

# expert between-polygon similarity, recomputed here for a side-by-side table
unique_polys, inverse, counts = np.unique(expert_poly_id, return_inverse=True, return_counts=True)
idx_a = rng.integers(0, len(expert_emb), N_PAIRS)
idx_b = rng.integers(0, len(expert_emb), N_PAIRS)
keep = expert_poly_id[idx_a] != expert_poly_id[idx_b]
norm_e = expert_emb / (np.linalg.norm(expert_emb, axis=1, keepdims=True) + 1e-12)
expert_between_sims = (norm_e[idx_a[keep]] * norm_e[idx_b[keep]]).sum(axis=1)

pd.DataFrame({
    "pair type": ["citizen-science, whole pilot extent", "citizen-science, only expert-containing blocks", "expert, between different polygons"],
    "n pairs": [len(cs_sims_full), len(cs_sims_restricted), len(expert_between_sims)],
    "mean cosine sim": [cs_sims_full.mean(), cs_sims_restricted.mean(), expert_between_sims.mean()],
    "median cosine sim": [np.median(cs_sims_full), np.median(cs_sims_restricted), np.median(expert_between_sims)],
}).round(4)


## 5. Mapa: onde estão as duas fontes no distrito-piloto


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(cs_x, cs_y, s=6, color="#2a78d6", alpha=0.5, label=f"citizen-science ({len(cs_x):,})")
ax.scatter(expert_x, expert_y, s=10, color="#eb6834", alpha=0.7, label=f"expert ({len(expert_x):,})")
ax.set_aspect("equal")
ax.set_xlabel("UTM 29N x (m)")
ax.set_ylabel("UTM 29N y (m)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")
ax.set_title("Where citizen-science and expert presences fall in the pilot extent")
fig.tight_layout()
plt.savefig("/tmp/diversity_map.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Leitura (2026-11-07)

**Não é geografia — é o tipo de habitat/local que cada fonte captura.**
O teste direto descarta a hipótese geográfica de forma clara: ciência
cidadã restrita aos mesmos 18 blocos de 10 km onde há especialistas
mantém a mesma similaridade (0,618) que tem no distrito inteiro
(0,614) — muito longe dos 0,778 entre polígonos de especialistas.
Estar na mesma área não torna a ciência cidadã tão homogénea quanto os
especialistas; logo a causa não é "estarem concentrados na mesma
zona".

**A composição de uso do solo (COS 2023) aponta para a explicação
real:** 55,0% dos píxeis de especialistas caem dentro de polígonos que
a própria COS já classifica como "floresta de acácias" — contra só
18,1% das presenças de ciência cidadã. A ciência cidadã espalha-se por
muito mais contextos: pinhal (32,6%), eucaliptal (20,4%), matos
(8,5%), e até artificial/estradas (4,7%) e água (0,5%), quase ausentes
do lado dos especialistas (0,3% e 0,0%). **Os especialistas mapearam
sobretudo manchas de acácia já grandes e reconhecíveis, enquanto a
ciência cidadã capta uma variedade muito maior de contextos de
ocorrência**, incluindo árvores isoladas perto de estradas, jardins e
orlas de outros tipos de floresta.

**O mapa confirma visualmente:** os especialistas formam alguns
focos apertados e bem separados entre si (nordeste do distrito, uma
mancha a meio, um ponto isolado a oeste) — não estão concentrados numa
única zona (daí a distância média entre pares ser *maior*, não menor,
do que a da ciência cidadã: 18,4 km vs. 10,9 km), mas cada foco é
internamente muito homogéneo. A ciência cidadã cobre uma área
contígua bem mais ampla, incluindo toda uma faixa a noroeste sem
nenhuma presença de especialista.

**Porque é que isto explica a curva plana:** se os especialistas
documentaram sobretudo o "caso fácil" (manchas grandes, já
identificáveis, parecidas com o que um mapa de uso do solo já
capturaria), e a ciência cidadã já cobre esse mesmo tipo de contexto
*e também* os casos mais marginais/variados, então adicionar mais
polígonos de especialistas tem pouco a acrescentar à diversidade que o
modelo já vê — não porque os dados de especialistas não tenham valor,
mas porque, **tal como amostrados neste piloto**, não preenchem uma
lacuna de diversidade que a ciência cidadã já não cubra.

A sobreposição de 55% com a classe "floresta de acácias" da própria
COS é coincidência ecológica — os dados de especialistas são
completamente independentes da COS (confirmado, 2026-11-07), não um
indício de circularidade entre as duas fontes.

**Fundamentação:** raciocínio meu a partir dos números desta
experiência; nenhuma literatura nova consultada.

**Em aberto:**
- Isto é um achado do recorte-piloto; não se sabe se um conjunto de
  especialistas mais disperso geograficamente (p. ex. cobrindo também
  a faixa noroeste só com ciência cidadã) mudaria a conclusão da curva.
  Implicação prática: se se voltar a recolher dados de especialistas,
  pode valer mais a pena visar locais/habitats subrepresentados do que
  mais locais do mesmo tipo.
- Resultado ainda não confirmado pelo orientador.

**Onde:** este notebook.
